# TRABAJO PRÁCTICO 1: FILTRADO E IDENTIFICACIÓN DE SISTEMAS (PREENTREGA)

**Cátedra:** Procesamiento Digital de Señales  
**Integrantes:** Ferreyra Florencia, González Tomás, Molina Lara y Scafati Jerónimo.  
**Fecha:** 08/06/2026

---

## 0. Importación de Librerías y Dependencias del Core

En lugar de duplicar código, importamos el motor matemático y de visualización desde nuestro módulo `functions.py` que se conecta directamente con el paquete `core/dsp` del proyecto. Esto asegura modularidad y consistencia en los algoritmos.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

from functions import (
    graficar_temp,
    graficar_frecuencias,
    rta_frecuencia,
    suma_tonos_puros,
    suma_musical,
    filtro_media_movil,
    filtro_peine,
    filtro_fir,
    filtrar_frecuencia_manual,
    generate_impulse,
    compute_frequency_response,
    plot_frequency_response,
    convolve_time,
    convolve_frequency,
    truncate_fir,
    plot_signal,
    plot_spectrum,
    load_audio,
    load_fir_coefficients,
    apply_fir,
    compute_fft,
    generate_pure_tones,
    add_white_noise
)

## 1. Respuesta al Impulso de los Filtros

La **respuesta al impulso** $h[n]$ es la salida de un sistema lineal e invariante en el tiempo (LTI) cuando la entrada es una delta de Dirac discreta (impulso unitario $\delta[n]$):

$$\delta[n] = \begin{cases} 1 & n = 0 \\ 0 & n \neq 0 \end{cases}$$

Matemáticamente, conocer $h[n]$ caracteriza por completo al sistema. Cualquier salida posterior $y[n]$ ante una entrada $x[n]$ arbitraria se puede calcular mediante la convolución lineal:

$$y[n] = x[n] * h[n] = \sum_{k=-\infty}^{\infty} x[k] h[n-k]$$

A continuación, generamos un impulso unitario y lo pasamos por los tres filtros diseñados:
1. **Media Móvil** (con ventana $M=8$ y $1$ pasada).
2. **Filtro Peine** (con coeficientes $b_0=0.5$, $b_1=0.3$, $b_2=0.2$).
3. **Filtro FIR** (coeficientes Hamming provistos por la cátedra).

In [ ]:
# 1. Generamos un impulso de longitud 100 con un delay de 5 para visualizarlo claramente
length = 100
impulse = generate_impulse(length, delay=5)

# 2. Pasamos el impulso por los distintos filtros para extraer h[n]
h_ma, _ = filtro_media_movil(impulse, m=8, p=1)
h_comb, _ = filtro_peine(impulse, a=0.5, b=0.3, c=0.2)
h_fir, _ = filtro_fir(impulse)

# 3. Graficamos las respuestas al impulso
t_axis = np.arange(length)
plt.figure(figsize=(10, 8))

plt.subplot(3, 1, 1)
plt.stem(t_axis, h_ma, basefmt=" ")
plt.title("Respuesta al Impulso: Media Móvil (M=8, passes=1)")
plt.xlabel("Muestras [n]")
plt.ylabel("Amplitud")
plt.grid(True)

plt.subplot(3, 1, 2)
plt.stem(t_axis, h_comb, basefmt=" ")
plt.title("Respuesta al Impulso: Filtro Peine (b0=0.5, b1=0.3, b2=0.2)")
plt.xlabel("Muestras [n]")
plt.ylabel("Amplitud")
plt.grid(True)

plt.subplot(3, 1, 3)
plt.plot(t_axis, h_fir[:length], color='green')
plt.title("Respuesta al Impulso: Filtro FIR de la Cátedra (Ventana Hamming)")
plt.xlabel("Muestras [n]")
plt.ylabel("Amplitud")
plt.grid(True)

plt.tight_layout()
plt.show()

### Análisis y Conclusiones de la Sección 1:
- **Filtro de Media Móvil (MA)**: Su respuesta al impulso es un pulso rectangular de amplitud constante $1/M$ (en este caso $1/8 = 0.125$) y de longitud exacta $M=8$ muestras a partir de la muestra del impulso. Esto coincide con su naturaleza de promedio aritmético de las últimas $M$ muestras.
- **Filtro Peine (Comb)**: Muestra exactamente 3 pulsos discretos no nulos correspondientes a los coeficientes $b_0=0.5$, $b_1=0.3$ y $b_2=0.2$ en los instantes del impulso original ($n=5$), un retardo de una muestra ($n=6$) y un retardo de dos muestras ($n=7$). Fuera de estos instantes, la respuesta es nula, verificando que es un filtro FIR causal de 3 coeficientes.
- **Filtro FIR de la Cátedra**: Presenta una respuesta al impulso más compleja y de mayor duración, con forma de función sinc suavizada por una ventana Hamming. Su simetría respecto al pico central indica que es un filtro de fase lineal.

## 2. Caracterización en Frecuencia

La respuesta en frecuencia del sistema $H(\omega)$ representa cómo se atenúa/amplifica la amplitud y cómo se desfasa cada componente sinusoidal en frecuencia al pasar por el filtro. Se determina mediante la relación entre las Transformadas de Fourier de la salida y de la entrada:

$$H(\omega) = \frac{Y(\omega)}{X(\omega)}$$

Al aplicar un impulso unitario $\delta[n]$ como entrada, su transformada de Fourier es $X(\omega) = 1$ para toda frecuencia. Por lo tanto, la respuesta en frecuencia del sistema es la transformada de Fourier de su respuesta al impulso $h[n]$:

$$H(\omega) = \mathcal{F}\{h[n]\}$$

Calculamos y graficamos el módulo (en decibelios, dB) y la fase (en radianes) de cada uno de los filtros.

In [ ]:
fs = 44100
# Con delay=0 para no introducir desfasajes lineales artificiales en la fase al graficar
imp_cal = generate_impulse(512, delay=0)

# Obtener las respuestas al impulso causales desde el origen
h_ma_c, _ = filtro_media_movil(imp_cal, m=8, p=1)
h_comb_c, _ = filtro_peine(imp_cal, a=0.5, b=0.3, c=0.2)
h_fir_c, _ = filtro_fir(imp_cal)

# Calcular H(w)
freqs_ma, H_ma = compute_frequency_response(imp_cal, h_ma_c, fs)
freqs_comb, H_comb = compute_frequency_response(imp_cal, h_comb_c, fs)
freqs_fir, H_fir = compute_frequency_response(imp_cal, h_fir_c, fs)

# Mostrar gráficos de respuesta en frecuencia usando la función del core dsp
fig_ma = plot_frequency_response(freqs_ma, H_ma, title="Respuesta en Frecuencia: Media Móvil (M=8)")
plt.show()

fig_comb = plot_frequency_response(freqs_comb, H_comb, title="Respuesta en Frecuencia: Filtro Peine")
plt.show()

fig_fir = plot_frequency_response(freqs_fir, H_fir, title="Respuesta en Frecuencia: Filtro FIR (Cátedra)")
plt.show()

### Análisis y Conclusiones de la Sección 2:
- **Filtro de Media Móvil**: Se comporta claramente como un filtro **pasabajos**. Sin embargo, es poco selectivo en la banda de paso y presenta nulos infinitos (caídas muy pronunciadas en el módulo) en frecuencias que son múltiplos enteros de $fs/M$ (para $fs=44100$ y $M=8$, ocurren en múltiplos de $5512.5\text{ Hz}$). En la fase se observan saltos de $\pi$ radianes en cada nulo espectral debido a los cambios de signo del lóbulo del sinc.
- **Filtro Peine**: Su respuesta en frecuencia presenta oscilaciones periódicas (valles y crestas de atenuación), asemejando la forma de un peine. Al ser de coeficientes positivos ($b_0=0.5, b_1=0.3, b_2=0.2$), la atenuación máxima ocurre cerca de la frecuencia Nyquist y presenta variaciones de fase suaves debido a la combinación de tres vectores temporales de retardo.
- **Filtro FIR de la Cátedra**: Presenta una selectividad excelente. Se comporta como un filtro **pasabajos plano** en la banda de paso con una atenuación abrupta en la frecuencia de corte (en torno a $1000\text{ Hz}$) y un excelente rechazo en la banda de parada. La fase en la banda de paso es perfectamente lineal, lo que significa que el filtro introduce un retardo de grupo constante y no distorsiona las fases relativas de los tonos.

## 3. Variación de Parámetros

Analizamos cómo varían los comportamientos espectrales al modificar los parámetros clave de los filtros:
1. **En la Media Móvil**: Variación del tamaño de ventana $M \in [3, 8, 20]$.
2. **En el Filtro Peine**: Variación de los coeficientes $b_0$, $b_1$, y $b_2$ para generar diferentes efectos de interferencia constructiva y destructiva en frecuencia.

In [ ]:
fs = 44100
imp_cal = generate_impulse(1024, delay=0)

# 1. Variación de M en Media Móvil
plt.figure(figsize=(10, 5))
for M in [3, 8, 20]:
    h_sweep, _ = filtro_media_movil(imp_cal, m=M, p=1)
    freqs, H = compute_frequency_response(imp_cal, h_sweep, fs=fs)
    mag_db = 20 * np.log10(np.clip(np.abs(H), 1e-15, None))
    plt.plot(freqs, mag_db, label=f"M = {M}")

plt.xscale('log')
plt.title("Media Móvil: Variación del tamaño de ventana M")
plt.xlabel("Frecuencia [Hz]")
plt.ylabel("Magnitud [dB]")
plt.ylim(-60, 5)
plt.grid(True, which='both')
plt.legend()
plt.show()

# 2. Variación de coeficientes en Filtro Peine
plt.figure(figsize=(10, 5))
casos_peine = [
    {"a": 1.0, "b": 0.0, "c": 0.0, "lbl": "Identidad (a=1)"},
    {"a": 0.5, "b": 0.5, "c": 0.0, "lbl": "Suma (a=0.5, b=0.5) -> Pasabajos (nulo en fs/2)"},
    {"a": 0.5, "b": -0.5, "c": 0.0, "lbl": "Resta (a=0.5, b=-0.5) -> Pasaaltos (nulo en DC)"},
    {"a": 0.5, "b": 0.0, "c": -0.5, "lbl": "Interferencia (a=0.5, c=-0.5) -> Nulo en fs/4"}
]

for caso in casos_peine:
    h_sweep, _ = filtro_peine(imp_cal, a=caso["a"], b=caso["b"], c=caso["c"])
    freqs, H = compute_frequency_response(imp_cal, h_sweep, fs=fs)
    mag_db = 20 * np.log10(np.clip(np.abs(H), 1e-15, None))
    plt.plot(freqs, mag_db, label=caso["lbl"])

plt.xscale('log')
plt.title("Filtro Peine: Variación de Coeficientes")
plt.xlabel("Frecuencia [Hz]")
plt.ylabel("Magnitud [dB]")
plt.ylim(-60, 5)
plt.grid(True, which='both')
plt.legend()
plt.show()

### Análisis y Conclusiones de la Sección 3:
- **Relación entre M y la Frecuencia de Corte en Media Móvil**:
  - A medida que **aumenta la ventana $M$**, la banda de paso del filtro se hace **más estrecha** (la frecuencia de corte disminuye, pasando de un filtro pasabajos ancho para $M=3$ a uno muy estrecho para $M=20$).
  - El primer nulo espectral ocurre en $f = fs/M$. Por ende, ventanas más grandes empujan los nulos y los lóbulos secundarios a frecuencias más bajas, aumentando la capacidad de suavizado del filtro temporalmente pero disminuyendo su ancho de banda.
- **Relación entre Coeficientes y Respuesta del Filtro Peine**:
  - Con $b_0=1, b_1=0, b_2=0$, el filtro peine es la función identidad (plano en 0 dB).
  - Al sumar coeficientes positivos adyacentes ($a=0.5, b=0.5$), el filtro suma muestras continuas y promedia las altas frecuencias, lo que causa un nulo en $fs/2$ (comportamiento pasabajos).
  - Al restar coeficientes adyacentes ($a=0.5, b=-0.5$), se atenúa la componente de corriente continua (nulo infinito en $0\text{ Hz}$ / DC) y se preservan las variaciones rápidas, actuando como un filtro pasaaltos.
  - Introducir retardos de 2 muestras con signos opuestos ($a=0.5, c=-0.5$) genera nulos periódicos intermedios (en este caso, en $fs/4 = 11025\text{ Hz}$). Esto demuestra cómo la ubicación de los coeficientes permite controlar la posición de los nulos espectrales del peine.

## 4. Señales de Prueba

Generamos las señales temporales de prueba para evaluar los filtros diseñados en escenarios prácticos:
1. **Mezcla de Tonos Ruidosa**: Suma de tres ondas senoidales ($500\text{ Hz}$, $1000\text{ Hz}$ y $5000\text{ Hz}$) con amplitudes respectivas de $1.0$, $0.5$ y $0.2$, contaminada con ruido blanco gaussiano para alcanzar una relación señal-ruido controlada de **$SNR = 15\text{ dB}$**.
2. **Audio Musical Ruidoso**: Señal de audio provista por la cátedra cargada desde disco, contaminada con ruido blanco gaussiano a **$SNR = 15\text{ dB}$**.

In [ ]:
fs = 44100
duracion = 1.0

# 1. Generación de mezcla de tonos limpios y adición de ruido SNR = 15 dB
tonos_limpios = generate_pure_tones(frequencies=[500.0, 1000.0, 5000.0], amplitudes=[1.0, 0.5, 0.2], fs=fs, duration=duracion)
tonos_ruidosos = add_white_noise(tonos_limpios, snr_db=15.0)

# 2. Carga de audio musical de la cátedra y adición de ruido SNR = 15 dB
# Usamos un bucle de búsqueda hacia arriba para encontrar el directorio raíz del proyecto
path = os.getcwd()
while path != '/' and not os.path.exists(os.path.join(path, "archivos")):
    path = os.path.dirname(path)
project_root = path

wav_path = os.path.join(project_root, "archivos", "musica_ruido_0.05.wav") # Archivo base de prueba
musica_limpia, fs_music = load_audio(wav_path)
if len(musica_limpia.shape) > 1:
    musica_limpia = musica_limpia[:, 0]  # Mono
musica_ruidosa = add_white_noise(musica_limpia, snr_db=15.0)

# 3. Visualización temporal (primeros ms para ver la forma de onda)
t_tonos = np.arange(len(tonos_limpios)) / fs
slice_tonos = slice(0, 400)

plt.figure(figsize=(12, 5))
plt.plot(t_tonos[slice_tonos], tonos_limpios[slice_tonos], label="Tonos Limpios", alpha=0.7)
plt.plot(t_tonos[slice_tonos], tonos_ruidosos[slice_tonos], label="Tonos Ruidosos (SNR=15 dB)", alpha=0.9, color='orange')
plt.title("Visualización Temporal: Mezcla de Tonos")
plt.xlabel("Tiempo [s]")
plt.ylabel("Amplitud")
plt.legend()
plt.grid(True)
plt.show()

# 4. Visualización espectral (espectros de magnitud de Fourier)
freqs_t, mags_t_limpio = compute_fft(tonos_limpios, fs)
_, mags_t_ruido = compute_fft(tonos_ruidosos, fs)

plt.figure(figsize=(12, 5))
plt.plot(freqs_t, mags_t_limpio, label="Espectro Limpio", alpha=0.7)
plt.plot(freqs_t, mags_t_ruido, label="Espectro Ruidoso", alpha=0.5, color='orange')
plt.title("Visualización Espectral: Espectro de Amplitud de Tonos")
plt.xlabel("Frecuencia [Hz]")
plt.ylabel("Magnitud")
plt.xlim(0, 6000)
plt.legend()
plt.grid(True)
plt.show()

### Análisis y Conclusiones de la Sección 4:
- En el **dominio del tiempo**, el ruido blanco añade variaciones aleatorias rápidas de pequeña amplitud (fluctuaciones caóticas) que distorsionan la forma de onda sinusoidal suave original de los tonos limpios.
- En el **dominio de la frecuencia**, el espectro limpio muestra únicamente 3 deltas (picos muy pronunciados) en las frecuencias exactas generadas ($500\text{ Hz}$, $1000\text{ Hz}$ y $5000\text{ Hz}$), con alturas proporcionales a sus amplitudes. El ruido blanco, en cambio, se distribuye de manera **plana y uniforme** (con magnitud promedio constante) en todo el espectro de frecuencia, elevando el "piso de ruido" general de la señal pero permitiendo aún ver los picos tonales debido a su alta potencia relativa.

## 5. Filtrado en Tiempo y Frecuencia

El filtrado digital mediante un filtro de respuesta al impulso finita $h[n]$ se puede realizar a través de dos caminos matemáticamente equivalentes:
1. **Convolución Lineal en el Tiempo**:
   $$y[n] = x[n] * h[n] = \sum_{k=0}^{M-1} h[k] x[n-k]$$
2. **Multiplicación en la Frecuencia (Teorema de la Convolución)**:
   $$Y(\omega) = X(\omega) \cdot H(\omega) \implies y[n] = \mathcal{F}^{-1}\{X(\omega) \cdot H(\omega)\}$$

Para que la convolución circular en frecuencia sea exactamente equivalente a la lineal sin distorsión por alias temporal, debemos realizar un zero-padding a ambos vectores ($x$ y $h$) hasta alcanzar una longitud mínima de $N_{\text{fft}} = L_x + L_h - 1$. 

A continuación, filtramos la señal de tonos ruidosos usando los dos métodos y medimos su error numérico de discrepancia.

In [ ]:
# Cargar los coeficientes FIR provistos por la cátedra para el filtrado
coefs_path = os.path.join(project_root, "archivos", "fir_hamming_1000Hz.npy")
h_fir = load_fir_coefficients(coefs_path)

# 1. Filtrado en tiempo usando convolución lineal completa
y_time = convolve_time(tonos_ruidosos, h_fir)

# 2. Filtrado en frecuencia usando convolución circular con padding N_fft
y_freq = convolve_frequency(tonos_ruidosos, h_fir)

# 3. Calcular la norma de la diferencia (discrepancia numérica)
error_norm = np.linalg.norm(y_time - y_freq)
print(f"Norma de la diferencia entre ambos métodos: {error_norm:.2e}")
print(f"¿Son numéricamente equivalentes?: {np.allclose(y_time, y_freq)}")

### Análisis y Conclusiones de la Sección 5:
- La norma de la diferencia entre los resultados del filtrado en tiempo y en frecuencia es del orden de **$10^{-15}$ o inferior** (cero práctico). Esto demuestra de forma empírica el **Teorema de la Convolución**.
- La pequeña diferencia residual se debe únicamente a los errores de redondeo de punto flotante de doble precisión (límite de precisión numérica de la máquina `float64`) acumulados en la Transformada Rápida de Fourier (FFT) y su inversa (IFFT) frente a la convolución directa en el dominio del tiempo.
- **Ventaja computacional**: Aunque el resultado es el mismo, para señales de gran longitud $N$ y filtros con muchos coeficientes $M$, el filtrado en frecuencia es órdenes de magnitud más rápido debido a la complejidad de la FFT ($O(N \log_2 N)$) frente a la de la convolución temporal directa ($O(N \cdot M)$).

## 6. Truncado de Coeficientes de Filtro FIR

El diseño de un filtro FIR ideal puede resultar en una respuesta al impulso con un número muy elevado de coeficientes. In la práctica, se trunca el filtro a una longitud menor $N < L$ para reducir los requisitos computacionales de memoria y potencia de cálculo, y el retardo del sistema.

Sin embargo, el truncado brusco de la respuesta al impulso equivale matemáticamente a multiplicar $h[n]$ por una ventana rectangular temporal, lo que en el dominio de la frecuencia se traduce en una convolución con una función sinc. Esto introduce:
1. **Ensanchamiento de la banda de transición** (el filtro es menos selectivo).
2. **Ripples espectrales (Fenómeno de Gibbs)** en la banda de paso y banda de parada, que no desaparecen al aumentar $N$ pero sí se concentran más cerca del borde de corte.

Evaluamos la respuesta en frecuencia del filtro FIR Hamming original frente a versiones truncadas a $N = [10, 50, 100]$ coeficientes.

In [ ]:
h_original = load_fir_coefficients(coefs_path)
L_orig = len(h_original)
fs = 44100

# Impulso largo para caracterizar el espectro con buena resolución
imp_eval = generate_impulse(2048, delay=0)

plt.figure(figsize=(12, 6))

# 1. Respuesta en frecuencia del original
y_orig = apply_fir(imp_eval, h_original)
_, H_orig = compute_frequency_response(imp_eval, y_orig, fs=fs)
mag_orig = 20 * np.log10(np.clip(np.abs(H_orig), 1e-15, None))
plt.plot(freqs_fir, mag_orig[:len(freqs_fir)], label=f"Original (L = {L_orig} coefs)", linewidth=2.5, color='black')

# 2. Respuestas para truncado N = 10, 50, 100
for N in [10, 50, 100]:
    h_trunc = truncate_fir(h_original, N)
    y_trunc = apply_fir(imp_eval, h_trunc)
    _, H_trunc = compute_frequency_response(imp_eval, y_trunc, fs=fs)
    mag_trunc = 20 * np.log10(np.clip(np.abs(H_trunc), 1e-15, None))
    plt.plot(freqs_fir, mag_trunc[:len(freqs_fir)], label=f"Truncado N = {N} coefs", alpha=0.8)

plt.xscale('log')
plt.title("Efecto del Truncado de Coeficientes en un Filtro FIR")
plt.xlabel("Frecuencia [Hz]")
plt.ylabel("Magnitud [dB]")
plt.ylim(-80, 10)
plt.grid(True, which='both')
plt.legend()
plt.show()

### Análisis y Conclusiones de la Sección 6:
- **Efecto de la reducción de $N$**:
  - Con **$N=10$**: El filtro pierde casi por completo su selectividad. La banda de transición es sumamente ancha, el corte en $1000\text{ Hz}$ es poco claro y la atenuación en la banda de parada es deficiente (solo llega a unos $-15\text{ dB}$ en las frecuencias altas). Además, se observan oscilaciones gigantescas.
  - Con **$N=50$**: La selectividad mejora significativamente. El comportamiento pasabajos es visible, con una pendiente de corte decente y el nivel de la banda de parada desciende a unos $-40\text{ dB}$.
  - Con **$N=100$**: La respuesta se aproxima de manera muy cercana a la del filtro original de $L=251$ coeficientes, logrando una banda de transición estrecha y una atenuación en la banda de parada de $-50\text{ dB}$.
- **Compromiso en Diseños Reales**:
  - **Mayor número de coeficientes ($N$ alto)**: Proporciona una respuesta en frecuencia excelente (banda de transición estrecha, menor ondulación, mayor atenuación de ruido), pero requiere más multiplicaciones por muestra, mayor memoria en hardware y añade mayor retardo temporal del filtro (igual a $(N-1)/2$ muestras).
  - **Menor número de coeficientes ($N$ bajo)**: Filtro de procesamiento muy rápido y con mínimo retardo de grupo, ideal para tiempo real de bajos recursos, pero a costa de atenuar menos el ruido y no recortar frecuencias con precisión.